In [ ]:
%%shell
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg


cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500

Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300

Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.LVIRpgJr1W/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.REZQLnezuh/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.cuaAZCTLcI/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 http://deb.debian.org/debian buster InRelease [122 kB]
Hit:2 http://archive.ubuntu.com/ubuntu 

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime

from selenium import webdriver
options = webdriver.ChromeOptions()
options.add_argument('--headless')        # Head-less 설정
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
import time 

#[CODE 1]
def CoffeeBean_store(result):
    CoffeeBean_URL = "https://www.coffeebeankorea.com/store/store.asp"
    wd = webdriver.Chrome('chromedriver', options=options)
             
    for i in range(1, 40):  #매장 수 만큼 반복
        wd.get(CoffeeBean_URL)
        time.sleep(1)  #웹페이지 연결할 동안 1초 대기
        try:
            wd.execute_script("storePop2(%d)" %i)
            time.sleep(1) #스크립트 실행 할 동안 1초 대기
            html = wd.page_source
            soupCB = BeautifulSoup(html, 'html.parser')
            store_name_h2 = soupCB.select("div.store_txt > h2")
            store_name = store_name_h2[0].string
            print(store_name)  #매장 이름 출력하기
            store_info = soupCB.select("div.store_txt > table.store_table > tbody > tr > td")
            store_address_list = list(store_info[2])
            store_address = store_address_list[0]
            store_phone = store_info[3].string
            result.append([store_name]+[store_address]+[store_phone])
        except:
            continue 
    return

#[CODE 0]
def main():
    result = []
    print('CoffeeBean store crawling >>>>>>>>>>>>>>>>>>>>>>>>>>')
    CoffeeBean_store(result)  #[CODE 1]
    
    CB_tbl = pd.DataFrame(result, columns=('store', 'address','phone'))
    CB_tbl.to_csv('./CoffeeBean.csv', encoding='utf-8', mode='w', index=True)

if __name__ == '__main__':
     main()

CoffeeBean store crawling >>>>>>>>>>>>>>>>>>>>>>>>>>


In [3]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support.ui import Select #select 태그 선택할때사용.
options = webdriver.ChromeOptions()
options.add_argument('--headless')        # Head-less 설정
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

def puradak_store(result):
    url = "https://puradakchicken.com/startup/store.asp"
    wd = webdriver.Chrome('chromedriver', options=options)
    wd.get(url)
    time.sleep(1)
    wd.find_element(By.CSS_SELECTOR, "#areaidx > option:nth-child(1)").click()
    # 매장 수 만큼 반복
    for page in range(1,5):
        if page > 1:
            nextBtn = wd.find_element(By.CLASS_NAME, "next")        # 다음 버튼
            wd.execute_script('arguments[0].click()', nextBtn)      # 다음 버튼 클릭

        for i in range(2, 22): #1페이지당 20개의 매장이 노출됨.
            try:
                time.sleep(1)  # 스크립트 실행 할 동안 1초 대기
                html = wd.page_source
                soupPRD = BeautifulSoup(html, 'html.parser')
                store_name_h2 = soupPRD.select(f"#result_search > li:nth-of-type({i}) > span > p.name")
                store_name = store_name_h2[0].string
                print(store_name)  # 매장 이름 출력
                store_doro = soupPRD.select(f"#result_search > li:nth-of-type({i}) > span > p.juso > span.doro")[0].string #도로명 주소
                store_phone = soupPRD.select(f"#result_search > li:nth-of-type({i}) > span > p.tel")[0].string #전화번호
                store_phone = store_phone.split()[2]           #연락처: 는 제외하고 전화번호만 가져오기
                print(store_phone)
                print(store_doro)
                result.append([store_name] + [store_doro] + [store_phone])

            except:
                continue
    return

def main():
    result = []
    print('Puradak store crawling')
    puradak_store(result)
    puradak_tbl = pd.DataFrame(result, columns=('store', 'address', 'phone'))
    puradak_tbl.to_csv('./Puradak.csv', encoding='utf-8-sig', mode='w', index=False)

if __name__ == '__main__':
    main()


Puradak store crawling


KeyboardInterrupt: ignored

Puradak store crawling >>>>>>>>>>>>>>>>>>>>>>>>>>
